# Problema 01: Edificio de 2 plantas (Shear Building)

## Relacion 3 - Sistemas de Multiples Grados de Libertad (MDOF)

**Objetivo:** Realizar el analisis modal completo de un edificio de 2 plantas modelado como "shear building" (forjados rigidos, pilares flexibles).

### Enunciado

Se tiene un edificio de 2 plantas con las siguientes caracteristicas:
- Masas concentradas en cada planta: $m_1 = m_2 = 1000$ kg
- Rigideces laterales de cada entreplanta: $k_1 = k_2 = 200$ kN/m

Se pide:
1. Construir las matrices de masa y rigidez del sistema
2. Calcular las frecuencias naturales y modos de vibracion
3. Determinar los factores de participacion modal y masas efectivas
4. Verificar la ortogonalidad de los modos
5. Simular la respuesta libre del sistema

## Datos del Problema

| Parametro | Valor | Unidad |
|-----------|-------|--------|
| $m_1$ | 1000 | kg |
| $m_2$ | 1000 | kg |
| $k_1$ | 200 | kN/m |
| $k_2$ | 200 | kN/m |

In [ ]:
# Importacion de librerias
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh

# Configuracion de graficos
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

# Datos del problema
m = np.array([1000.0, 1000.0])  # kg - masas por planta
k = np.array([200e3, 200e3])     # N/m - rigideces por entreplanta

print("DATOS DEL PROBLEMA")
print("="*40)
print(f"Masas: m1 = {m[0]:.0f} kg, m2 = {m[1]:.0f} kg")
print(f"Rigideces: k1 = {k[0]/1e3:.0f} kN/m, k2 = {k[1]/1e3:.0f} kN/m")

## 1. Construccion de las Matrices del Sistema

Para un modelo de "shear building" con $n$ plantas:

**Matriz de masa (diagonal):**
$$\mathbf{M} = \begin{bmatrix} m_1 & 0 \\ 0 & m_2 \end{bmatrix}$$

**Matriz de rigidez (tridiagonal):**
$$\mathbf{K} = \begin{bmatrix} k_1 + k_2 & -k_2 \\ -k_2 & k_2 \end{bmatrix}$$

In [ ]:
def shear_building(m, k):
    """
    Construye las matrices M y K para un shear building.
    
    Parametros:
        m: array de masas por planta [m1, m2, ..., mn]
        k: array de rigideces por entreplanta [k1, k2, ..., kn]
    
    Retorna:
        M: matriz de masa (diagonal)
        K: matriz de rigidez (tridiagonal)
    """
    n = len(m)
    
    # Matriz de masa (diagonal)
    M = np.diag(m)
    
    # Matriz de rigidez (tridiagonal)
    K = np.zeros((n, n))
    for i in range(n):
        if i == 0:
            K[i, i] = k[i] + (k[i+1] if i+1 < n else 0)
        elif i == n-1:
            K[i, i] = k[i]
        else:
            K[i, i] = k[i] + k[i+1]
        
        if i < n-1:
            K[i, i+1] = -k[i+1]
            K[i+1, i] = -k[i+1]
    
    return M, K

# Construir matrices
M, K = shear_building(m, k)

print("MATRICES DEL SISTEMA")
print("="*40)
print("\nMatriz de masa M (kg):")
print(M)
print("\nMatriz de rigidez K (N/m):")
print(K)

## 2. Analisis Modal: Frecuencias y Modos

El problema de valores propios generalizado es:
$$\mathbf{K} \boldsymbol{\phi} = \omega^2 \mathbf{M} \boldsymbol{\phi}$$

Donde:
- $\omega_n$ son las frecuencias naturales circulares (rad/s)
- $\boldsymbol{\phi}_n$ son los modos de vibracion

In [ ]:
# Resolver problema de valores propios generalizado
# K*phi = omega^2 * M * phi
eigenvalues, eigenvectors = eigh(K, M)

# Frecuencias naturales
omega = np.sqrt(eigenvalues)  # rad/s
freq = omega / (2 * np.pi)     # Hz
T = 1 / freq                   # s (periodo)

# Modos de vibracion (normalizados masa-unidad)
phi = eigenvectors

print("FRECUENCIAS NATURALES")
print("="*40)
print(f"\nModo 1: omega_1 = {omega[0]:.3f} rad/s")
print(f"        f_1 = {freq[0]:.3f} Hz")
print(f"        T_1 = {T[0]:.3f} s")
print(f"\nModo 2: omega_2 = {omega[1]:.3f} rad/s")
print(f"        f_2 = {freq[1]:.3f} Hz")
print(f"        T_2 = {T[1]:.3f} s")
print(f"\nRatio de frecuencias: omega_2/omega_1 = {omega[1]/omega[0]:.3f}")

In [ ]:
print("MODOS DE VIBRACION")
print("="*40)
print("(Normalizados masa-unidad: phi^T M phi = 1)")
print(f"\nModo 1: phi_1 = [{phi[0,0]:.4f}, {phi[1,0]:.4f}]")
print(f"Modo 2: phi_2 = [{phi[0,1]:.4f}, {phi[1,1]:.4f}]")

# Normalizar para visualizacion (maximo = 1)
phi_norm = np.zeros_like(phi)
for i in range(phi.shape[1]):
    phi_norm[:, i] = phi[:, i] / np.max(np.abs(phi[:, i]))

print(f"\nModos normalizados (max=1):")
print(f"Modo 1: [{phi_norm[0,0]:.3f}, {phi_norm[1,0]:.3f}]")
print(f"Modo 2: [{phi_norm[0,1]:.3f}, {phi_norm[1,1]:.3f}]")

## 3. Factores de Participacion y Masas Efectivas

El **factor de participacion modal** para excitacion en la base es:
$$\Gamma_n = \frac{\boldsymbol{\phi}_n^T \mathbf{M} \mathbf{r}}{\boldsymbol{\phi}_n^T \mathbf{M} \boldsymbol{\phi}_n}$$

Donde $\mathbf{r} = [1, 1]^T$ es el vector de influencia (excitacion horizontal uniforme).

La **masa efectiva modal** es:
$$M_{ef,n} = \Gamma_n^2 \cdot (\boldsymbol{\phi}_n^T \mathbf{M} \boldsymbol{\phi}_n) = \frac{(\boldsymbol{\phi}_n^T \mathbf{M} \mathbf{r})^2}{\boldsymbol{\phi}_n^T \mathbf{M} \boldsymbol{\phi}_n}$$

In [ ]:
# Vector de influencia (excitacion horizontal en la base)
r = np.array([1.0, 1.0])

# Factores de participacion modal
n_modes = phi.shape[1]
Gamma = np.zeros(n_modes)
M_ef = np.zeros(n_modes)

for n in range(n_modes):
    phi_n = phi[:, n]
    L_n = phi_n @ M @ r  # Masa modal excitable
    m_n = phi_n @ M @ phi_n  # Masa modal generalizada
    
    Gamma[n] = L_n / m_n
    M_ef[n] = L_n**2 / m_n

# Masa total
M_total = np.sum(m)
M_ef_pct = M_ef / M_total * 100

print("FACTORES DE PARTICIPACION MODAL")
print("="*40)
print(f"\nGamma_1 = {Gamma[0]:.4f}")
print(f"Gamma_2 = {Gamma[1]:.4f}")

print("\nMASAS EFECTIVAS MODALES")
print("="*40)
print(f"\nM_ef,1 = {M_ef[0]:.1f} kg ({M_ef_pct[0]:.1f}%)")
print(f"M_ef,2 = {M_ef[1]:.1f} kg ({M_ef_pct[1]:.1f}%)")
print(f"\nSuma = {np.sum(M_ef):.1f} kg (M_total = {M_total:.1f} kg)")
print(f"Verificacion: {np.sum(M_ef_pct):.1f}% (debe ser 100%)")

## 4. Verificacion de Ortogonalidad

Los modos de vibracion son ortogonales respecto a M y K:
$$\boldsymbol{\phi}_i^T \mathbf{M} \boldsymbol{\phi}_j = 0 \quad (i \neq j)$$
$$\boldsymbol{\phi}_i^T \mathbf{K} \boldsymbol{\phi}_j = 0 \quad (i \neq j)$$

Para modos normalizados masa-unidad:
$$\boldsymbol{\phi}_i^T \mathbf{M} \boldsymbol{\phi}_i = 1$$
$$\boldsymbol{\phi}_i^T \mathbf{K} \boldsymbol{\phi}_i = \omega_i^2$$

In [ ]:
phi1, phi2 = phi[:, 0], phi[:, 1]

print("VERIFICACION DE ORTOGONALIDAD")
print("="*40)

# Ortogonalidad respecto a M
ort_M = phi1 @ M @ phi2
print(f"\nphi_1^T M phi_2 = {ort_M:.2e} (debe ser ~ 0)")

# Ortogonalidad respecto a K
ort_K = phi1 @ K @ phi2
print(f"phi_1^T K phi_2 = {ort_K:.2e} (debe ser ~ 0)")

# Normalizacion masa-unidad
print(f"\nphi_1^T M phi_1 = {phi1 @ M @ phi1:.6f} (debe ser = 1)")
print(f"phi_2^T M phi_2 = {phi2 @ M @ phi2:.6f} (debe ser = 1)")

# Verificacion de frecuencias
print(f"\nphi_1^T K phi_1 = {phi1 @ K @ phi1:.2f} (omega_1^2 = {omega[0]**2:.2f})")
print(f"phi_2^T K phi_2 = {phi2 @ K @ phi2:.2f} (omega_2^2 = {omega[1]**2:.2f})")

## 5. Respuesta Libre del Sistema

La respuesta libre (sin amortiguamiento) se puede expresar mediante superposicion modal:
$$\mathbf{x}(t) = \sum_{n=1}^{N} \boldsymbol{\phi}_n q_n(t)$$

Donde las coordenadas modales:
$$q_n(t) = A_n \cos(\omega_n t) + B_n \sin(\omega_n t)$$

Con condiciones iniciales:
$$A_n = \boldsymbol{\phi}_n^T \mathbf{M} \mathbf{x}(0)$$
$$B_n = \frac{\boldsymbol{\phi}_n^T \mathbf{M} \dot{\mathbf{x}}(0)}{\omega_n}$$

In [ ]:
def respuesta_libre(phi, omega, M, x0, v0, t):
    """
    Calcula la respuesta libre de un sistema MDOF sin amortiguamiento.
    
    Parametros:
        phi: matriz modal (columnas = modos)
        omega: frecuencias naturales
        M: matriz de masa
        x0: desplazamientos iniciales
        v0: velocidades iniciales
        t: vector de tiempo
    
    Retorna:
        x: matriz de respuesta [n_dof x n_time]
    """
    n_dof = len(x0)
    n_time = len(t)
    x = np.zeros((n_dof, n_time))
    
    for n in range(n_dof):
        phi_n = phi[:, n]
        A_n = phi_n @ M @ x0
        B_n = phi_n @ M @ v0 / omega[n]
        
        q_n = A_n * np.cos(omega[n] * t) + B_n * np.sin(omega[n] * t)
        x += np.outer(phi_n, q_n)
    
    return x

# Condiciones iniciales: desplazamiento inicial en planta 1
x0 = np.array([0.01, 0.0])  # 10 mm en planta 1
v0 = np.array([0.0, 0.0])   # Sin velocidad inicial

# Vector de tiempo (2 segundos)
t = np.linspace(0, 2, 500)

# Calcular respuesta
x_t = respuesta_libre(phi, omega, M, x0, v0, t)

print(f"Condiciones iniciales:")
print(f"  x_1(0) = {x0[0]*1000:.1f} mm")
print(f"  x_2(0) = {x0[1]*1000:.1f} mm")

## 6. Visualizacion de Resultados

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# Panel 1: Modos de vibracion
ax1 = axes[0]
plantas = [0, 1, 2]
modo1_plot = np.concatenate([[0], phi_norm[:, 0]])
modo2_plot = np.concatenate([[0], phi_norm[:, 1]])

ax1.plot(modo1_plot, plantas, 'b-o', linewidth=2, markersize=10, 
         label=f'Modo 1 (f={freq[0]:.2f} Hz)')
ax1.plot(modo2_plot, plantas, 'r-s', linewidth=2, markersize=10, 
         label=f'Modo 2 (f={freq[1]:.2f} Hz)')
ax1.axvline(x=0, color='k', linestyle='--', linewidth=0.5)
ax1.set_xlabel('Desplazamiento normalizado')
ax1.set_ylabel('Planta')
ax1.set_yticks([0, 1, 2])
ax1.set_yticklabels(['Base', 'Planta 1', 'Planta 2'])
ax1.legend(loc='upper left')
ax1.set_title('Modos de vibracion')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(-1.5, 1.5)

# Panel 2: Masas efectivas
ax2 = axes[1]
modos_labels = ['Modo 1', 'Modo 2']
colors = ['steelblue', 'coral']
bars = ax2.bar(modos_labels, M_ef_pct, color=colors, edgecolor='black')
ax2.axhline(y=90, color='green', linestyle='--', linewidth=1.5, 
            label='90% (minimo sismico)')
ax2.set_ylabel('Masa efectiva (%)')
ax2.set_title('Participacion modal')
ax2.legend()
ax2.set_ylim(0, 100)
for bar, pct in zip(bars, M_ef_pct):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f'{pct:.1f}%', ha='center', fontsize=11, fontweight='bold')

# Panel 3: Respuesta libre
ax3 = axes[2]
ax3.plot(t, x_t[0, :] * 1000, 'b-', linewidth=1.5, label='x1(t) - Planta 1')
ax3.plot(t, x_t[1, :] * 1000, 'r-', linewidth=1.5, label='x2(t) - Planta 2')
ax3.set_xlabel('Tiempo (s)')
ax3.set_ylabel('Desplazamiento (mm)')
ax3.set_title('Respuesta libre (x1(0)=10 mm)')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Interpretacion de Resultados

### Frecuencias naturales
- **Modo 1** (fundamental): $f_1 \approx 1.27$ Hz. Este modo tiene la menor frecuencia y domina la respuesta sismica.
- **Modo 2**: $f_2 \approx 3.33$ Hz. El ratio $\omega_2/\omega_1 \approx 2.62$ indica modos bien separados.

### Modos de vibracion
- **Modo 1**: Ambas plantas se mueven en la misma direccion (en fase). Es el modo fundamental.
- **Modo 2**: Las plantas se mueven en direcciones opuestas (antifase). Mayor deformacion de entrepiso.

### Masas efectivas
- El **Modo 1** participa con ~90% de la masa total, lo que significa que domina la respuesta sismica.
- El **Modo 2** solo contribuye con ~10%.
- Para analisis sismico, con el Modo 1 ya se supera el 90% requerido por la mayoria de normativas.

### Respuesta libre
- La respuesta muestra el "batido" tipico de sistemas de 2DOF.
- La combinacion de dos frecuencias produce patrones complejos de vibracion.
- Sin amortiguamiento, la vibracion continua indefinidamente.